In [2]:
# Import necessary libraries.
from sklearn.linear_model import LogisticRegression # Logistic Regression model
from sklearn.ensemble import RandomForestClassifier # Random Forest classifier
from sklearn.model_selection import train_test_split, GridSearchCV # Modules for splitting train/test data and for Grid Search
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif # Modules for feature selection (Select top K, Variance Threshold, ANOVA F-value)
from sklearn.tree import DecisionTreeClassifier # Decision Tree classifier
from sklearn.metrics import roc_auc_score, fbeta_score, make_scorer # ROC AUC score, F-beta score, create custom scorer
from xgboost import XGBClassifier # XGBoost classifier
import shap # SHAP(SHapley Additive exPlanations) library (explaining model predictions)
import matplotlib.pyplot as plt # Library for data visualization

import pandas as pd # Library for data manipulation and analysis
import numpy as np # Library for numerical calculations
import datetime as dt # Library for date and time handling
import json # Library for JSON data handling

In [3]:
# Module for handling warning messages
import warnings 

# Ignore only the 'use_label_encoder' warning.
warnings.filterwarnings("ignore")

#### prepare "data/initial_dataset.p"

In [4]:
# # .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3

# # Specify the file path
# file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.xlsx'

# # Read the Excel file into a DataFrame
# # By default, it reads the first sheet.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [5]:
# # .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress

# # Specify the file path
# file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.xlsx'

# # Read the Excel file into a DataFrame
# # By default, it reads the first sheet.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)


In [6]:
# read *.p
pickle_file_path_1 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'
data_row_1 = pd.read_pickle(pickle_file_path_1)
pickle_file_path_2 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'
data_row_2 = pd.read_pickle(pickle_file_path_2)


In [7]:
# Keep only the necessary columns

# Specify the file path
file_path = 'data/cols_to_keep.csv'

# Read the CSV file into a DataFrame
cols_to_keep_df = pd.read_csv(file_path)

cols_to_keep = cols_to_keep_df.iloc[:, 0].tolist()

data_row_1 = data_row_1[cols_to_keep]


In [8]:
# data_row <= data_row1 data_row2

# Select only the columns to join from data_row_2
columns_to_join = ['DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP',
                   'BG pass/fail']

# Create a subset DataFrame of data_row_2 with the selected columns
data_row_2_subset = data_row_2[columns_to_join]

# Merge the selected columns from data_row_2 into data_row_1 using the 'DevID' join key
data_row = pd.merge(data_row_1, data_row_2_subset, on='DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP', how='left')


In [9]:
initial_dataset = data_row.copy() # Copy the original dataset

In [10]:
# prepare for target

initial_dataset['Pass/Fail_pass'] = ((initial_dataset['soft_bin of FT1'] == 1) &
                   (initial_dataset['soft_bin of FT2'] == 1) &
                   (initial_dataset['soft_bin'] == 1)).astype(int)


In [11]:
# prepare for base model

initial_dataset['band gap dpat'] = initial_dataset['BG pass/fail'].apply(lambda x: 'bandGapFail' if x == 'impossible wafer' else 'ok for band gap')

# Create a dictionary for column name changes
new_column_names = {
    'wafer_id': 'WAFER_NO',
    'DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP': 'DevID'
}

# Rename the columns using the .rename() method (using inplace=True to apply directly to the original DataFrame)
initial_dataset.rename(columns=new_column_names, inplace=True)


In [12]:
initial_dataset


,WF of 1103959_69_1133529_cp1,ROW of 1103959_69_1133529_cp1,COL of 1103959_69_1133529_cp1,DevID,S of 1103959_69_1133529_cp1,F:E of 1103959_69_1133529_cp1,FAILING_BINOUTS(1) of 1103959_69_1133529_cp1,HH_CONT_OUT of 1103959_69_1133529_cp1,HH_CONT_CL of 1103959_69_1133529_cp1,HH_CONT_CM of 1103959_69_1133529_cp1,...,EEPROM_T_SCAL_G_BIAS[],EEPROM_V4[],EEPROM_BANK8_D[],EEPROM_BANK8_C[],EEPROM_BANK9_D[],EEPROM_BANK9_C[],EEPROM_LOCKPAT_check[],BG pass/fail,Pass/Fail_pass,band gap dpat
0,2,17,35,[110395 17 35],8,.:.,NaN,-0.3747,-0.3812,-0.3783,...,4.0,122.0,35.0,199.0,18.0,123.0,83.0,impossible wafer,1,bandGapFail
1,2,19,38,[110395 19 38],8,.:.,NaN,-0.3849,-0.3934,-0.3894,...,3.0,140.0,35.0,199.0,18.0,123.0,83.0,NaN,1,ok for band gap
2,2,-7,63,[110395 -7 63],8,.:.,NaN,-0.3859,-0.3815,-0.3792,...,243.0,128.0,35.0,199.0,18.0,123.0,83.0,impossible wafer,1,bandGapFail
3,2,13,70,[110395 13 70],8,.:.,NaN,-0.3928,-0.3833,-0.3811,...,4.0,96.0,36.0,71.0,18.0,123.0,83.0,NaN,1,ok for band gap
4,2,33,57,[110395 33 57],8,.:.,NaN,-0.3875,-0.3796,-0.3768,...,243.0,148.0,35.0,199.0,18.0,123.0,83.0,NaN,1,ok for band gap
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4531,2,31,63,[113352 31 63],8,.:.,NaN,-0.3798,-0.4050,-0.4015,...,243.0,112.0,35.0,198.0,18.0,123.0,83.0,NaN,1,ok for band gap
4532,2,-7,29,[113352 -7 29],8,.:.,NaN,-0.3965,-0.4209,-0.4189,...,242.0,128.0,35.0,199.0,18.0,123.0,83.0,pass,1,ok for band gap
4533,2,20,32,[113352 20 32],8,.:.,NaN,-0.3673,-0.4128,-0.4070,...,243.0,128.0,35.0,199.0,18.0,123.0,83.0,NaN,1,ok for band gap
4534,2,35,60,[113352 35 60],8,.:.,NaN,-0.3786,-0.4056,-0.4240,...,243.0,140.0,35.0,199.0,18.0,123.0,83.0,NaN,1,ok for band gap


In [13]:
# Define a list of columns to drop
columns_to_drop = [
    'soft_bin of FT1',
    'soft_bin of FT2',
    'soft_bin',
    'BG pass/fail'
]

# Drop columns (use inplace=True to modify the original DataFrame)
# Or create a new DataFrame with processed_dataset = processed_dataset.drop(...)
initial_dataset.drop(columns=columns_to_drop, inplace=True)


In [14]:
# Rename the columns
initial_dataset.rename(columns={'x_pos': 'X', 'y_pos': 'Y'}, inplace=True)


In [15]:
# Calculate the Radius column
# The np.sqrt() function calculates the square root of each element.
initial_dataset['Radius'] = np.sqrt(initial_dataset['X']**2 + initial_dataset['Y']**2)
initial_dataset.to_pickle("data/initial_dataset.p")


#### scnarios


In [16]:
# import vars and function

from algos.algos import *
from config.config import *


In [17]:
import copy


##### preprocess_dataset


In [18]:
preprocessed_dataset = preprocess_dataset(initial_dataset)




     데이터셋 전처리 중...
     전처리 완료!



##### create_train_and_test_data


In [19]:
split_parameter = copy.deepcopy(split_parameter_default)
train_data, test_data, split_parameter_info = create_train_test_data(preprocessed_dataset, split_parameter)




##############################################################################################################################
# 3) Create Train/Test Split (훈련/테스트 데이터 분할) 
##############################################################################################################################

     훈련 및 테스트 데이터셋 생성 중...
     - 분할 전 필터링 미적용.
     - Feature Generation 미적용.

     - 분할 전 훈련 데이터 클래스 분포: {0.0: 3546, 1.0: 71}
     - 샘플링 미적용


##### Use user-specified thresholds - feature selection and feature importance calculation


##### select_feature


In [20]:
feature_selector_params_var = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_var["filter_methods"]["apply_variance_filter"] = True
feature_selector_params_var["filter_methods"]["var_threshold"] = 0.00
feature_selection_info_var = select_feature(train_data, feature_selector_params_var)

feature_selector_params_licor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_licor["filter_methods"]["apply_target_linear_corr_filter"] = True
feature_selector_params_licor["filter_methods"]["target_linear_corr_threshold"] = 0.00
feature_selection_info_licor = select_feature(train_data, feature_selector_params_licor)

feature_selector_params_xicor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_xicor["filter_methods"]["apply_target_xicor_filter"] = True
feature_selector_params_xicor["filter_methods"]["target_xicor_threshold"] = 0.00
feature_selection_info_xicor = select_feature(train_data, feature_selector_params_xicor)

feature_selector_params_sfm = copy.deepcopy(feature_selector_params_sfm_default)
feature_selector_params_sfm["params"]["estimator"]["params"]["n_estimators"] = 250 # max = len(train_data.columns) - 1
feature_selector_params_sfm["params"]["threshold"] = "0*median"
feature_selection_info_sfm = select_feature(train_data, feature_selector_params_sfm)

feature_selection_infos = {
     "var" : feature_selection_info_var,
     "licor" : feature_selection_info_licor,
     "xicor" : feature_selection_info_xicor,
     "model" : feature_selection_info_sfm
}
for fileter_name, feature_selection_info in feature_selection_infos.items():
    print("# of feature:", feature_selection_info["final_feature_count"], ",  filter: ", feature_selection_info["feature_selector_name"], fileter_name)




--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 분산 필터링 후 남은 피처 수: 1401

피처 선택 결과가 'data/result/jsons\feature_selection_info_250903_073803_e1272ab5.json' 파일에 저장되었습니다.

- 최종 피처 수: 1401

--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 타겟 선형 상관관계 필터링 후 남은 피처 수: 1650

피처 선택 결과가 'data/result/jsons\feature_selection_info_250903_073803_a8a8f9ef.json' 파일에 저장되었습니다.

- 최종 피처 수: 1650

--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 타겟 Xi Cor 필터링 후 남은 피처 수: 1650

피처 선택 결과가 'data/result/jsons\feature_selection_info_250903_073803_78ad0ed6.json' 파일에 저장되었습니다.

- 최종 피처 수: 1650

--- 피처 선택기: SFM ---
--- SFM 선택기 완료 ---
남은 피처 수: 1650

피처 선택 결과가 'data/result/jsons\feature_selection_info_250903_073805_435e46a9.json' 파일에 저장되었습니다.

- 최종 피처 수: 1650
# of feature: 1401 ,  filter:  FeatureFilter var
# of feature: 1650 ,  filter:  FeatureFilter licor
# of feature: 1650 ,  filter:  FeatureFilter xicor
# of feature: 1650 ,  filter:  SFM model


In [21]:
# Summarize feature selector importance information

features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for fileter_name, feature_selection_info in feature_selection_infos.items():
    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    features_values_df = pd.DataFrame(
        list(features_values.items()), 
        columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    )

    features_values_dfs = pd.merge(
        features_values_dfs,
        features_values_df,
        how='outer',
        left_on='feature_name',
        right_on='feature_name'
        )


In [22]:
# # Feature selection results by method: dic

feature_selection_info


{'feature_selector_name': 'SFM',
 'params': {'threshold': '0*median',
  'estimator': {'name': 'RandomForestClassifier',
   'params': {'n_estimators': 250, 'max_depth': 12}}},
 'filter_methods': 'apply_SelectFromModel_filter',
 'initial_feature_count': 1650,
 'final_feature_count': 1650,
 'final_features': ['X',
  'Y',
  'Radius',
  'AC_COIL_FACTOR of 1103959_69_1133529_YPP',
  'AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5',
  'AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5_YPP_QPP',
  'AC_COIL_FACTOR of 1103959_69_1133592_RPP',
  'AC_GAIN_32 of 1103959_69_1133529_cp1',
  'AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP',
  'AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP',
  'AC_GAIN_32 of 1103959_69_1133529_cp1p5',
  'AC_GAIN_32 of 1103959_69_1133592_QPP',
  'AC_GAIN_32 of 1103959_69_JPP',
  'AC_GAIN_33 of 1103959_69_1133529_cp1',
  'AC_GAIN_33 of 1103959_69_1133529_cp1_cp1p5_YPP',
  'AC_GAIN_33 of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP',
  'AC_GAIN_33 of 1103959_69_1133529_

In [23]:
# # Feature selection results summary data by method: df

features_values_dfs


,feature_name,FeatureFilter_variance,FeatureFilter_target_linear_correlation,FeatureFilter_target_xicor_correlation,SFM_importances
0,AC_COIL_FACTOR of 1103959_69_1133529_YPP,2.951574e-33,0.021211,0.255277,0.000000
1,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,2.951574e-33,0.021211,0.255277,0.000000
2,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5...,2.951574e-33,0.021211,0.255277,0.000000
3,AC_COIL_FACTOR of 1103959_69_1133592_RPP,2.951574e-33,0.021211,0.255277,0.000000
4,AC_GAIN_32 of 1103959_69_1133529_cp1,0.000000e+00,NaN,0.456388,0.000000
...,...,...,...,...,...
1646,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1_cp1...,1.000277e+00,0.017764,0.505893,0.000395
1647,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1p5,1.000277e+00,0.026030,0.396025,0.000935
1648,Zb_V_OBVOL_VSS_L of 1103959_69_1133592_QPP,1.000277e+00,0.001384,0.447422,0.000143
1649,Zb_V_OBVOL_VSS_L of 1103959_69_JPP,1.000277e+00,0.067453,0.604254,0.000389


In [24]:
# feature select test : pipeline
## Apply the selected feature set feature_selection_info['final_features'] list to the modeling pipeline and return the modeling results

def pl_fs_test(
        train_data,
        feature_selection_info,
        train_parameters,
        test_data
        ):
    
    trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results


##### Use user-specified thresholds - feature selection > performance check: Use a simple model pipeline


In [25]:
# Apply the model based on the specified threshold and summarize the results
# feature select test - submit and summary result

usr_pl_fs_test_result_ftpn_df = pd.DataFrame()
usr_pl_fs_test_result_features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for fileter_name, feature_selection_info in feature_selection_infos.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_test(
            train_data,
            feature_selection_info,
            train_parameters_list_default["rf_cv"],
            test_data
            )

    dict_ftpn = metrics["dict_ftpn"]
    
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'filter_methods' : feature_selection_info["filter_methods"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'feature_selection_info_json_path' : feature_selection_info["feature_selection_info_json_path"]
    }
    
    usr_pl_fs_test_result_ftpn_df = pd.concat([usr_pl_fs_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)

    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    features_values_df = pd.DataFrame(
        list(features_values.items()), 
        columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    )

    usr_pl_fs_test_result_features_values_dfs = pd.merge(
        usr_pl_fs_test_result_features_values_dfs,
        features_values_df,
        how='outer',
        left_on='feature_name',
        right_on='feature_name'
        )


      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
    Best F2 (class=1) score (CV): 0.2258

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.7172 with F2 score: 0.6153
      Calculation of the ROC curve...
      Calculation done
      Scoring...
      Scoring done

      Creating the metrics...
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.2240

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.6970 with F2 score: 0.5909
      Calculation of the ROC curve...
      Calculation done
      Sco

In [26]:
# Model performance summary of the feature selection set - based on user feature selection threshold

usr_pl_fs_test_result_ftpn_df


,fn,fp,tn,tp,feature_selector_name,filter_methods,initial_feature_count,final_feature_count,feature_selection_info_json_path
0,5,180,707,13,FeatureFilter,"{'apply_variance_filter': True, 'var_threshold...",1650,1401,data/result/jsons\feature_selection_info_25090...
1,6,184,703,12,FeatureFilter,"{'apply_variance_filter': False, 'var_threshol...",1650,1650,data/result/jsons\feature_selection_info_25090...
2,6,184,703,12,FeatureFilter,"{'apply_variance_filter': False, 'var_threshol...",1650,1650,data/result/jsons\feature_selection_info_25090...
3,6,184,703,12,SFM,apply_SelectFromModel_filter,1650,1650,data/result/jsons\feature_selection_info_25090...


In [27]:
# Model performance summary of the feature selection set + applied columns from user feature selection threshold (for reference)

usr_pl_fs_test_result_features_values_dfs


,feature_name,FeatureFilter_variance,FeatureFilter_target_linear_correlation,FeatureFilter_target_xicor_correlation,SFM_importances
0,AC_COIL_FACTOR of 1103959_69_1133529_YPP,2.951574e-33,0.021211,0.255277,0.000000
1,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,2.951574e-33,0.021211,0.255277,0.000000
2,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5...,2.951574e-33,0.021211,0.255277,0.000000
3,AC_COIL_FACTOR of 1103959_69_1133592_RPP,2.951574e-33,0.021211,0.255277,0.000000
4,AC_GAIN_32 of 1103959_69_1133529_cp1,0.000000e+00,NaN,0.456388,0.000000
...,...,...,...,...,...
1646,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1_cp1...,1.000277e+00,0.017764,0.505893,0.000395
1647,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1p5,1.000277e+00,0.026030,0.396025,0.000935
1648,Zb_V_OBVOL_VSS_L of 1103959_69_1133592_QPP,1.000277e+00,0.001384,0.447422,0.000143
1649,Zb_V_OBVOL_VSS_L of 1103959_69_JPP,1.000277e+00,0.067453,0.604254,0.000389


##### Use feature importance information - optimal selection > performance check: Use a simple model pipeline


###### The optimal selection method uses the XGBClassifier model and GridSearchCV for searching for optimal values


In [28]:
feature_importance_df = features_values_dfs.copy()


In [29]:
### Apply

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import fbeta_score, make_scorer, confusion_matrix
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import uuid


In [30]:
# train_data


In [31]:
# Select the target as the rightmost column
target_df = train_data.iloc[:, -1]
# The rest of the columns are features
features_df = train_data.iloc[:, :-1]

X_train, X_test, y_train, y_test = train_test_split(features_df, target_df, test_size=0.2, random_state=42)


In [32]:
# run_optimization_for_feature_importance : a function to select features based on a specific importance column and find the optimal model

def run_optimization_for_feature_importance(train_data, target_data, feature_importance_df, importance_column, k_percentiles):
    """
    A function to select features based on a specific importance column and find the optimal model.
    
    Parameters:
    - train_data (pd.DataFrame): Training data
    - target_data (pd.Series): Target data
    - feature_importance_df (pd.DataFrame): DataFrame containing feature importance information
    - importance_column (str): The column name to determine the importance rank
    - k_percentiles (list): List of candidate percentiles for the number of features to select (e.g., [0.05, 0.1, 0.25, 0.5])

    Returns:
    - pd.DataFrame: Optimized feature importance information
    - pd.DataFrame: Model performance summary information
    """
    f2_scorer = make_scorer(fbeta_score, beta=2.0)
    
    # Select the top K features based on the values in the importance column
    sorted_features = feature_importance_df.sort_values(
        by=importance_column, ascending=False
    )['feature_name']
    
    # Convert percentiles to the actual number of features
    n_features_total = len(sorted_features)
    k_values = [max(1, int(n_features_total * p)) for p in k_percentiles]
    
    best_k = k_values[0]
    best_score = -1.0
    best_pipeline = None
    selected_feature_list = []
    # ⭐️ Variable to store the optimal percentile value
    best_k_percentile = k_percentiles[0]

    for i, k in enumerate(k_values):
        top_k_features = sorted_features.head(k).tolist()
        
        # Prepare the dataset with only the optimal features
        X_train_filtered = train_data[top_k_features]
        
        # Model training pipeline (direct feature selection instead of SelectKBest)
        pipeline = Pipeline([
            ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
        ])
        
        param_grid = {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 5]
        }
        
        grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
        grid_search.fit(X_train_filtered, target_data)

        # Evaluate the performance for the current K
        if grid_search.best_score_ > best_score:
            best_score = grid_search.best_score_
            best_k = k
            best_pipeline = grid_search.best_estimator_
            selected_feature_list = top_k_features
            # ⭐️ Update the best percentile
            best_k_percentile = k_percentiles[i]

    # Generate feature importance and performance information for the optimal model
    best_xgb_model = best_pipeline.named_steps['model']
    
    feature_info = pd.DataFrame({
        'feature_name': train_data.columns
    })
    feature_info['is_selected'] = feature_info['feature_name'].isin(selected_feature_list)
    feature_info['importance_column'] = importance_column
    feature_importances = {name: 0 for name in train_data.columns}
    
    # Assign importance scores only to the selected features
    for i, importance in enumerate(best_xgb_model.feature_importances_):
        if i < len(selected_feature_list):
            feature_importances[selected_feature_list[i]] = importance
        
    feature_info['importance_score'] = feature_info['feature_name'].map(feature_importances)
    feature_info['feature_value_by_importance_column'] = feature_info['feature_name'].map(
        feature_importance_df.set_index('feature_name')[importance_column]
    )

    # Final performance evaluation with the test data
    y_pred = best_pipeline.predict(X_test[selected_feature_list])
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    performance_summary = pd.DataFrame([{
        'fn': fn,
        'fp': fp,
        'tn': tn,
        'tp': tp,
        'feature_selector_name': importance_column,
        'initial_feature_count': X_train.shape[1],
        'final_feature_count': len(selected_feature_list),
        'f2_score': fbeta_score(y_test, y_pred, beta=2.0),
        'importance_column': importance_column,
        # ⭐️ Add the optimal percentile column
        'best_k_percentile': best_k_percentile
    }])

    return feature_info, performance_summary


In [ ]:
# Repeat optimization for each importance column and accumulate results
# Save the importance column names from the 2nd column onwards, excluding the feature name column
importance_cols = feature_importance_df.columns[1:].tolist()

# ⭐️ Change to a list of percentile candidates: defined by the user
k_percentiles = [0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9]

all_feature_infos = []
all_performance_summaries = []

for col in importance_cols:
    print(f"\n--- Running optimization based on {col} column ---")
    feat_info, perf_summary = run_optimization_for_feature_importance(
        X_train, y_train, feature_importance_df, col, k_percentiles
    )
    all_feature_infos.append(feat_info)
    all_performance_summaries.append(perf_summary)

final_feature_info_df = pd.concat(all_feature_infos, ignore_index=True)
final_feature_performance_summary_df = pd.concat(all_performance_summaries, ignore_index=True)

# 4. Print results and save to CSV
print("\n--- Final accumulated feature importance information ---")
print(final_feature_info_df.head(10))
final_feature_info_df.to_csv('final_feature_info.csv', index=False)

print("\n--- Final accumulated model performance summary ---")
print(final_feature_performance_summary_df)
final_feature_performance_summary_df.to_csv('final_feature_performance_summary.csv', index=False)



--- Running optimization based on FeatureFilter_variance column ---

--- Running optimization based on FeatureFilter_target_linear_correlation column ---

--- Running optimization based on FeatureFilter_target_xicor_correlation column ---

--- Running optimization based on SFM_importances column ---


In [ ]:
# Automatic (optimal) feature selection result data: df
final_feature_info_df


,feature_name,is_selected,importance_column,importance_score,feature_value_by_importance_column
0,X,True,FeatureFilter_variance,0.000000,1.000277e+00
1,Y,True,FeatureFilter_variance,0.007534,1.000277e+00
2,Radius,True,FeatureFilter_variance,0.003558,1.000277e+00
3,AC_COIL_FACTOR of 1103959_69_1133529_YPP,False,FeatureFilter_variance,0.000000,2.951574e-33
4,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,False,FeatureFilter_variance,0.000000,2.951574e-33
...,...,...,...,...,...
6595,FAILING_BINOUTS(1) of 1103959_69_JPP_GbGbSbSbPoPo,False,SFM_importances,0.000000,7.015850e-05
6596,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,False,SFM_importances,0.000000,0.000000e+00
6597,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,False,SFM_importances,0.000000,6.139330e-07
6598,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,False,SFM_importances,0.000000,0.000000e+00


In [ ]:
# Automatic (optimal) feature selection result data summary: df
final_feature_performance_summary_df

,fn,fp,tn,tp,feature_selector_name,initial_feature_count,final_feature_count,f2_score,importance_column,best_k_percentile
0,14,7,703,0,FeatureFilter_variance,1650,1320,0.0,FeatureFilter_variance,0.8
1,14,7,703,0,FeatureFilter_target_linear_correlation,1650,165,0.0,FeatureFilter_target_linear_correlation,0.1
2,14,7,703,0,FeatureFilter_target_xicor_correlation,1650,1485,0.0,FeatureFilter_target_xicor_correlation,0.9
3,14,6,704,0,SFM_importances,1650,825,0.0,SFM_importances,0.5


##### Use feature importance information - optimal selection > performance check: Use a simple model pipeline

In [ ]:
# Automatic (optimal) feature selection result data: df
final_feature_info_df


,feature_name,is_selected,importance_column,importance_score,feature_value_by_importance_column
0,X,True,FeatureFilter_variance,0.000000,1.000277e+00
1,Y,True,FeatureFilter_variance,0.007534,1.000277e+00
2,Radius,True,FeatureFilter_variance,0.003558,1.000277e+00
3,AC_COIL_FACTOR of 1103959_69_1133529_YPP,False,FeatureFilter_variance,0.000000,2.951574e-33
4,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,False,FeatureFilter_variance,0.000000,2.951574e-33
...,...,...,...,...,...
6595,FAILING_BINOUTS(1) of 1103959_69_JPP_GbGbSbSbPoPo,False,SFM_importances,0.000000,7.015850e-05
6596,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,False,SFM_importances,0.000000,0.000000e+00
6597,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,False,SFM_importances,0.000000,6.139330e-07
6598,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,False,SFM_importances,0.000000,0.000000e+00


In [ ]:
# Prepare performance check pipeline input: feature_selection_results
feature_selection_results = {}

# Iterate line by line
for idx, row in final_feature_performance_summary_df.iterrows():
    feature_selection_result = {}
    feature_selection_result["feature_selector_idx"] = idx
    feature_selection_result["feature_selector_name"] = row['feature_selector_name']
    feature_selection_result["initial_feature_count"] = row['initial_feature_count']
    feature_selection_result["final_feature_count"] = row['final_feature_count']
    feature_selection_result.setdefault("Params", {})["f2_score"] = row['f2_score']
    feature_selection_result.setdefault("Params", {})["best_k_percentile"] = row['best_k_percentile']
    
    feature_name_list = final_feature_info_df[
        (final_feature_info_df["importance_column"] == row['feature_selector_name']) &
        (final_feature_info_df["is_selected"] == True)
    ]["feature_name"].tolist()
    feature_selection_result["final_features"] = feature_name_list

    feature_selection_results[idx] = feature_selection_result

In [ ]:
# final_feature_performance_summary_df

In [ ]:
# feature_selection_results

In [ ]:
# Optimal feature set performance check
# feature select test - submit and summary result

trained_model_set = {}
feature_importance_set = pd.DataFrame()
forecast_dataset_set = pd.DataFrame()
train_dataset_proba_set = pd.DataFrame()
best_threshold_set = pd.DataFrame()
roc_data_set = pd.DataFrame()
auc_score_set = pd.DataFrame()
train_dataset_metrics_set = pd.DataFrame()
metrics_set = pd.DataFrame()
results_set = pd.DataFrame()

opt_pl_fs_test_result_ftpn_df = pd.DataFrame()
opt_pl_fs_test_result_features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for feature_selector_idx, feature_selection_info in feature_selection_results.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_test(
            train_data,
            feature_selection_info,
            train_parameters_list_default["rf_cv"],
            test_data
            )

    trained_model_set[feature_selector_idx] = trained_model

    feature_importance["feature_selector_idx"] = feature_selector_idx
    feature_importance_set = pd.concat([feature_importance_set, feature_importance], ignore_index=True)

    forecast_dataset_df = pd.DataFrame({
        "feature_selector_idx": [feature_selector_idx] * len(forecast_dataset),
        "forecast_prob": forecast_dataset
    })
    forecast_dataset_set = pd.concat([forecast_dataset_set, forecast_dataset_df], ignore_index=True)
    
    train_dataset_proba["feature_selector_idx"] = feature_selector_idx
    train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

    train_dataset_proba["feature_selector_idx"] = feature_selector_idx
    train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

    best_threshold_df = pd.DataFrame({
    "feature_selector_idx": [feature_selector_idx],
    "best_threshold": [best_threshold]
    })
    best_threshold_set = pd.concat([best_threshold_set, best_threshold_df], ignore_index=True)

    roc_data["feature_selector_idx"] = feature_selector_idx
    roc_data_set = pd.concat([roc_data_set, roc_data], ignore_index=True)

    auc_score_df = pd.DataFrame({
    "feature_selector_idx": [feature_selector_idx],
    "auc_score": [auc_score]
    })
    auc_score_set = pd.concat([auc_score_set, auc_score_df], ignore_index=True)

    train_dataset_metrics["feature_selector_idx"] = feature_selector_idx
    train_dataset_metrics_set = pd.concat([train_dataset_metrics_set, train_dataset_metrics], ignore_index=True)

    metrics_dict = {
        "f1_score": metrics["f1_score"],
        "recall": metrics["recall"],
        "precision": metrics["precision"],
        "accuracy": metrics["accuracy"],
        "auc_score": metrics["auc_score"],
        "tp": metrics["dict_ftpn"]["tp"],
        "tn": metrics["dict_ftpn"]["tn"],
        "fp": metrics["dict_ftpn"]["fp"],
        "fn": metrics["dict_ftpn"]["fn"],
        "number_of_predictions": metrics["number_of_predictions"],
        "number_of_good_predictions": metrics["number_of_good_predictions"],
        "number_of_false_predictions": metrics["number_of_false_predictions"],
        "feature_selector_idx": feature_selector_idx  # 현재 필터 이름 추가
    }
    metrics_df = pd.DataFrame([metrics_dict])
    metrics_set = pd.concat([metrics_set, metrics_df], ignore_index=True)

    results["feature_selector_idx"] = feature_selector_idx
    results_set = pd.concat([results_set, results], ignore_index=True)

    dict_ftpn = metrics["dict_ftpn"]
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_idx' : feature_selection_info["feature_selector_idx"],
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'final_feature_selector_f2score' : feature_selection_info["Params"]["f2_score"],
        'final_feature_selector_best_k_percentile' : feature_selection_info["Params"]["best_k_percentile"]
    }
    
    opt_pl_fs_test_result_ftpn_df = pd.concat([opt_pl_fs_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)
    features_values = feature_importance[["Features", "Importance"]].rename(columns={"Importance": "Importance(model)"+"_"+feature_selection_info["feature_selector_name"] })
    opt_pl_fs_test_result_features_values_dfs = pd.merge(
        opt_pl_fs_test_result_features_values_dfs,
        features_values,
        how='left',
        left_on='feature_name',
        right_on='Features'
    ).drop('Features', axis=1) # The .drop() method is used to drop the column

import pickle
import os

save_path = "data/result/select_feature"
os.makedirs(save_path, exist_ok=True)

dict_vars = [
    (trained_model_set, "trained_model_set")
]

df_vars = [
    (feature_importance_set, "feature_importance_set"),
    (forecast_dataset_set, "forecast_dataset_set"),
    (train_dataset_proba_set, "train_dataset_proba_set"),
    (best_threshold_set, "best_threshold_set"),
    (roc_data_set, "roc_data_set"),
    (auc_score_set, "auc_score_set"),
    (train_dataset_metrics_set, "train_dataset_metrics_set"),
    (metrics_set, "metrics_set"),
    (results_set, "results_set"),
    (opt_pl_fs_test_result_ftpn_df, "opt_pl_fs_test_result_ftpn_df"),
    (opt_pl_fs_test_result_features_values_dfs, "opt_pl_fs_test_result_features_values_dfs")
]

for var, name in dict_vars:
    file_path = os.path.join(save_path, f"{name}.pickle")
    with open(file_path, "wb") as f:
        pickle.dump(var, f)
    print(f"Saved {name} to {file_path}")

for var, name in df_vars:
    file_path = os.path.join(save_path, f"{name}.csv")
    var.to_csv(file_path, index=False)
    print(f"Saved {name} to {file_path}")

      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.2192

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.6970 with F2 score: 0.6132
     Calculation of the ROC curve...
     Calculation done
     Scoring...
     Scoring done

     Creating the metrics...
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.1966

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.7980 with F2 score: 0.5182
     Calculation of the ROC curve...
     Calculation done
     Scoring...
    

In [ ]:
opt_pl_fs_test_result_ftpn_df


,fn,fp,tn,tp,feature_selector_idx,feature_selector_name,initial_feature_count,final_feature_count,final_feature_selector_f2score,final_feature_selector_best_k_percentile
0,6,178,709,12,0,FeatureFilter_variance,1650,1320,0.0,0.8
1,8,199,688,10,1,FeatureFilter_target_linear_correlation,1650,165,0.0,0.1
2,7,158,729,11,2,FeatureFilter_target_xicor_correlation,1650,1485,0.0,0.9
3,8,132,755,10,3,SFM_importances,1650,825,0.0,0.5


In [ ]:
opt_pl_fs_test_result_features_values_dfs

,feature_name,Importance(model)_FeatureFilter_variance,Importance(model)_FeatureFilter_target_linear_correlation,Importance(model)_FeatureFilter_target_xicor_correlation,Importance(model)_SFM_importances
0,X,0.000000,NaN,0.000000,0.000000
1,Y,0.008596,NaN,0.000000,NaN
2,Radius,0.004065,NaN,0.008714,0.006605
3,AC_COIL_FACTOR of 1103959_69_1133529_YPP,NaN,NaN,NaN,NaN
4,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,NaN,NaN,NaN,NaN
...,...,...,...,...,...
1646,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,NaN,NaN,0.000000,NaN
1647,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,NaN,NaN,0.000000,NaN
1648,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,NaN,NaN,0.000000,NaN
1649,band gap dpat_ok for band gap,NaN,NaN,0.000000,NaN


In [ ]:
train_dataset_proba_set

In [1]:
test_data

NameError: name 'test_data' is not defined

###### Model Parameter optimization with optuna

In [62]:
# feature select test : pipeline
## Apply the selected feature set feature_selection_info['final_features'] list to the modeling pipeline and return the modeling results

def pl_fs_test(
        train_data,
        feature_selection_info,
        train_parameters,
        test_data
        ):
    
    trained_model, feature_importance, train_parameters_info = train_model_rf_optuna(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results


In [63]:
# Optimal feature set performance check
# feature select test - submit and summary result

opt_pl_fs_test_result_ftpn_df_rf_optuna = pd.DataFrame()
opt_pl_fs_test_result_features_values_dfs_rf_optuna = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

feature_importance_set = pd.DataFrame()

for fileter_name, feature_selection_info in feature_selection_results.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_test(
            train_data,
            feature_selection_info,
            train_parameters_list_default["rf_optuna"],
            test_data
            )

    dict_ftpn = metrics["dict_ftpn"]
    
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_idx' : feature_selection_info["feature_selector_idx"],
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'final_feature_selector_f2score' : feature_selection_info["Params"]["f2_score"],
        'final_feature_selector_best_k_percentile' : feature_selection_info["Params"]["best_k_percentile"]
    }
    
    opt_pl_fs_test_result_ftpn_df_rf_optuna = pd.concat([opt_pl_fs_test_result_ftpn_df_rf_optuna, pd.DataFrame([new_row])], ignore_index=True)
    features_values = feature_importance[["Features", "Importance"]].rename(columns={"Importance": "Importance(model)"+"_"+feature_selection_info["feature_selector_name"] })

    opt_pl_fs_test_result_features_values_dfs_rf_optuna = pd.merge(
        opt_pl_fs_test_result_features_values_dfs_rf_optuna,
        features_values,
        how='left',
        left_on='feature_name',
        right_on='Features'
    ).drop('Features', axis=1) # The .drop() method is used to drop the column


[I 2025-09-02 09:55:25,812] A new study created in memory with name: no-name-cfa92ab1-9980-41ae-ab01-c0d9c9721fbb


     Training the Random Forest model with Optuna hyperparameter tuning...



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-09-02 09:55:35,494] Trial 0 finished with value: 0.15363062594816232 and parameters: {'n_estimators': 219, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight_multiplier': 1289}. Best is trial 0 with value: 0.15363062594816232.
[I 2025-09-02 09:56:10,851] Trial 1 finished with value: 0.18363941025912284 and parameters: {'n_estimators': 239, 'max_depth': 11, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': 0.5, 'class_weight_multiplier': 1417}. Best is trial 1 with value: 0.18363941025912284.
[I 2025-09-02 09:56:18,075] Trial 2 finished with value: 0.04502888415931894 and parameters: {'n_estimators': 242, 'max_depth': 29, 'min_samples_split': 19, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight_multiplier': 544}. Best is trial 1 with value: 0.18363941025912284.
[I 2025-09-02 09:56:37,231] Trial 3 finished with value: 0.19208381401085584 and parameters: {'n_estimators': 132, 'max_depth': 10, 'min_samples_

[I 2025-09-02 10:09:34,631] A new study created in memory with name: no-name-a3178f80-a54c-4e28-becb-8bc3dacbc863


...

[I 2025-09-02 10:13:28,836] Trial 0 finished with value: 0.2095985749815151 and parameters: {'n_estimators': 181, 'max_depth': 14, 'min_samples_split': 17, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'class_weight_multiplier': 3489}. Best is trial 0 with value: 0.2095985749815151.
[I 2025-09-02 10:13:33,209] Trial 1 finished with value: 0.16241313737643645 and parameters: {'n_estimators': 279, 'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'class_weight_multiplier': 2048}. Best is trial 0 with value: 0.2095985749815151.
[I 2025-09-02 10:14:05,138] Trial 2 finished with value: 0.0402864044168392 and parameters: {'n_estimators': 142, 'max_depth': 28, 'min_samples_split': 6, 'min_samples_leaf': 10, 'max_features': 0.5, 'class_weight_multiplier': 256}. Best is trial 0 with value: 0.2095985749815151.
[I 2025-09-02 10:15:16,952] Trial 3 finished with value: 0.014925373134328356 and parameters: {'n_estimators': 185, 'max_depth': 29, 'min_samples_sp

[I 2025-09-02 10:27:00,672] A new study created in memory with name: no-name-3f313189-92b1-4b4b-8d08-2c03a568d81f


     Creating the metrics...
     Training the Random Forest model with Optuna hyperparameter tuning...



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-09-02 10:28:08,313] Trial 0 finished with value: 0.17844455563947612 and parameters: {'n_estimators': 296, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 0.8, 'class_weight_multiplier': 1286}. Best is trial 0 with value: 0.17844455563947612.
[I 2025-09-02 10:29:14,162] Trial 1 finished with value: 0.06702802581319896 and parameters: {'n_estimators': 241, 'max_depth': 20, 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_features': 0.8, 'class_weight_multiplier': 2368}. Best is trial 0 with value: 0.17844455563947612.
[I 2025-09-02 10:30:07,975] Trial 2 finished with value: 0.17013194698927075 and parameters: {'n_estimators': 240, 'max_depth': 12, 'min_samples_split': 18, 'min_samples_leaf': 10, 'max_features': 0.8, 'class_weight_multiplier': 3104}. Best is trial 0 with value: 0.17844455563947612.
[I 2025-09-02 10:31:06,590] Trial 3 finished with value: 0.06695959942954102 and parameters: {'n_estimators': 300, 'max_depth': 22, 'min_samples_spli